# 📊 DSAI 413 — Data Exploration
Explore the MIMIC-CXR dataset: structure, columns, report text, images.

**Run this locally or on Kaggle (no GPU needed).**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# Try to load via our module
from src.data.load_dataset import load_dataset, find_dataset_files, get_dataset_stats

## 1. Find & Load Data
If running on Kaggle, the dataset should be added as an input dataset.

In [ ]:
# Adjust this path based on your environment
# Local: '../data/raw'
# Kaggle: '/kaggle/input/mimic-cxr-dataset'

DATA_PATH = '../data/raw'
# DATA_PATH = '/kaggle/input/mimic-cxr-dataset'  # Uncomment for Kaggle

files = find_dataset_files(DATA_PATH)
print('CSV files:', [str(f) for f in files['csv_files']])
print('Image files:', len(files['image_files']))

In [ ]:
df = load_dataset(DATA_PATH)
stats = get_dataset_stats(df)
df.head()

## 2. Explore Report Text

In [ ]:
# Report length distribution
df['report_length'] = df['report_text'].astype(str).str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['report_length'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Report Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')

word_counts = df['report_text'].astype(str).str.split().str.len()
axes[1].hist(word_counts, bins=50, color='coral', edgecolor='black')
axes[1].set_title('Report Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Avg report length: {df["report_length"].mean():.0f} chars')
print(f'Avg word count: {word_counts.mean():.0f} words')

In [ ]:
# Sample reports
for i in range(3):
    print(f'\n{"="*60}')
    print(f'Report {i}:')
    print(df['report_text'].iloc[i][:500])

## 3. Preprocess & Create Subset

In [ ]:
from src.data.preprocess import preprocess_pipeline

train_df, test_df, full_df = preprocess_pipeline(df, subset_size=1000)
print(f'\nTrain: {len(train_df)}, Test: {len(test_df)}, Full subset: {len(full_df)}')

## 4. Generate QA Dataset

In [ ]:
from src.data.create_qa_dataset import generate_qa_dataset, generate_qa_pairs_for_report

# Quick test on one report
sample_report = full_df['report_text'].iloc[0]
pairs = generate_qa_pairs_for_report(sample_report, 'test_img.jpg', 'test_001')
print(f'Generated {len(pairs)} QA pairs from sample report:')
for p in pairs:
    print(f'  Q: {p["question"]}')
    print(f'  A: {p["answer"][:100]}...')
    print(f'  Cat: {p["category"]}\n')

In [ ]:
# Generate full QA dataset
qa_df = generate_qa_dataset(full_df)
print(f'\nTotal QA pairs: {len(qa_df)}')
print(f'\nCategory distribution:')
print(qa_df['category'].value_counts())

In [ ]:
# Visualize QA category distribution
fig, ax = plt.subplots(figsize=(10, 6))
qa_df['category'].value_counts().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('QA Category Distribution')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()